# LLM Classification Finetuning: submission (Fase 4)

Notebook de envío. Corre sin internet: codifica el test con `bge-small-en-v1.5`, calcula las
features y predice con el LightGBM entrenado localmente (`python src/train.py --embeddings`).

Entradas que debe tener el notebook (panel *Input*):
- la competencia `llm-classification-finetuning`;
- un dataset con el contenido de `artifacts/` (`lgbm.txt`, `projections.npz`, `data.py`,
  `features.py`, `predict.py`);
- los pesos de `BAAI/bge-small-en-v1.5` (una carpeta con `modules.json`).

Activa una GPU (Accelerator) para que la codificación sea rápida.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

INPUT = Path("/kaggle/input")
ARTIFACTS = next(INPUT.rglob("lgbm.txt")).parent
BGE = next(p.parent for p in INPUT.rglob("modules.json") if "bge" in str(p).lower())
sys.path.insert(0, str(ARTIFACTS))

from data import load_test
from predict import predict

COLS = ["prompt", "response_a", "response_b"]
print("artifacts:", ARTIFACTS)
print("bge model:", BGE)

In [ ]:
TEST_DIR = next(p.parent for p in INPUT.rglob("test.csv") if "llm-classification-finetuning" in str(p))
test = load_test(TEST_DIR)
print(TEST_DIR, test.shape)
test[["id"]].head(3)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(str(BGE), device=device)
model.max_seq_length = 512
if device == "cuda":
    model.half()

texts = pd.unique(pd.concat([test[c] for c in COLS]))
print(f"{device}: {len(test):,} rows, {len(texts):,} unique texts")
vectors = model.encode(list(texts), batch_size=128, normalize_embeddings=True,
                       show_progress_bar=False, convert_to_numpy=True).astype(np.float16)
index = {t: i for i, t in enumerate(texts)}
emb = {c: vectors[test[c].map(index).to_numpy()] for c in COLS}

In [ ]:
probs = predict(test, emb, ARTIFACTS)

submission = pd.DataFrame(probs, columns=["winner_model_a", "winner_model_b", "winner_tie"])
submission.insert(0, "id", test["id"].to_numpy())
assert len(submission) == len(test) and np.allclose(probs.sum(axis=1), 1)
submission.to_csv("submission.csv", index=False)
submission.head()